# Predict concept transfer from steering (Figure 5a)

This notebook screens arbitrary animal/tree preference prompts with `allenai/OLMo-2-1124-7B-Instruct`. It extracts a paired system-vs-no-system $v_{teacher}$ on 1,024 diverse semantic instructions (not number sequences), then uses the repository's Figure-5-style clean steering peak as the predictor.

The result is a one-sided heuristic, not a guarantee or calibrated probability: Figure 5a showed that concepts with zero steering failed to transfer, while a positive steering rate did not specify how many training epochs transfer would require. OLMo-2 is also an extrapolation; the paper tested OLMo-3. The off-topic guard deliberately tests the clean *preference* component: a vector that obeys the literal instruction to mention the target in everything can be rejected as token puppeting.

## Setup

From the repository root, install the project and launch this notebook in the same environment:

```bash
uv sync
uv run --with jupyterlab jupyter lab notebooks/figure5_concept_transfer.ipynb
```

The BF16 checkpoint is about 15 GB, so a 20-24 GB GPU is a practical minimum. The default SDPA backend needs no separate FlashAttention install.

In [ ]:
from subliminal.transfer_predictor import (
    ANIMALS,
    TREES,
    ANIMAL_SYSTEM_PROMPT,
    TREE_SYSTEM_PROMPT,
    TransferPredictionConfig,
    TransferPredictor,
    default_target_form,
    predict_transfer,
)

print(ANIMAL_SYSTEM_PROMPT)
print(TREE_SYSTEM_PROMPT)
print(f"{len(ANIMALS)} animals, {len(TREES)} trees")
print("Animal targets:", [default_target_form(x, "animals") for x in ANIMALS])
print("Tree targets:", [default_target_form(x, "trees") for x in TREES])

## Configure once

With `RUN_FULL=True`, the configuration retains the repository's full zoo settings: layers `(6, 12, 18, 24, 30)`, alphas `(1, 2, 4, 8)`, 100 samples per evaluation prompt, raw vectors, and the local repo's inclusive 10% negative/off-topic gates. That is roughly 180,000 sampled completions per concept, or 5.04 million for the supplied 9-animal and 19-tree lists. The grid was copied from OLMo-3 and may need widening for OLMo-2. Grid points, vectors, and final results are cached for resumable reruns.

`RUN_FULL=False` is the safe Run-All default: it uses a deliberately noisy quick configuration and one concept from each list only. Toggle it only when you intend to run the full experiment.

In [ ]:
RUN_FULL = False
config = TransferPredictionConfig() if RUN_FULL else TransferPredictionConfig.quick()

predictor = TransferPredictor(config)  # model loading is lazy and happens only once

## One-call animal screen

Edit `my_animals` or pass `ANIMALS` directly. The bundled singular labels render to the exact supplied targets, including `wolfs`.

In [ ]:
my_animals = list(ANIMALS if RUN_FULL else ANIMALS[:1])
animal_predictions = predict_transfer(my_animals, domain="animals", predictor=predictor)
animal_predictions

## One-call tree screen

Tree evaluation uses a matched set of 50 favorite-tree, 20 least-favorite-tree, and the same 20 off-topic controls. The labels `bristlecone` and `eucalyptus` render as the exact targets `bristlecone pines` and `eucalyptus trees`.

In [ ]:
my_trees = list(TREES if RUN_FULL else TREES[:1])
tree_predictions = predict_transfer(my_trees, domain="trees", predictor=predictor)
tree_predictions

## Work with the result

`clean_peak_pos_rate` is the Figure-5 predictor score. `passes_zero_steering_screen` preserves the paper's absolute nonzero-rate test, while `steering_effect_detected` checks that the clean peak actually improves on the unsteered baseline. By default, zero steering or no lift is `unlikely`; a positive clean rate with positive lift is `not_ruled_out`, so `predicted_transfer` is `False` or `None` rather than using an invented success cutoff. To request a binary added heuristic, explicitly set `transfer_threshold` (and optionally `minimum_steering_lift`) in the config.

In [ ]:
# pandas is optional; the object itself is a normal list of detailed dictionaries.
animal_df = animal_predictions.to_dataframe()
animal_df.sort_values("clean_peak_pos_rate", ascending=False)

In [ ]:
# Full layer/alpha grid and exact prompt used for one concept:
example = animal_predictions[0]
print(example["system_prompt"])
example["grid"]

## Custom spellings or multiword concepts

Use `target_forms` for the exact text substituted into `{target}` and `aliases` for additional accepted generations. Because the bundled Figure-5 prompts request one-word answers, a multiword concept needs a one-word alias (as below) or custom prompt sets. Custom prompt banks and domains can also be passed to `predict_transfer`.

In [ ]:
RUN_CUSTOM_EXAMPLE = False
custom_predictions = (
    predict_transfer(
        ["mountain ash"],
        domain="trees",
        predictor=predictor,
        target_forms={"mountain ash": "mountain ashes"},
        # Bundled prompts request one-word answers, so supply a one-word alias.
        aliases={"mountain ash": ["rowan"]},
    )
    if RUN_CUSTOM_EXAMPLE
    else "Set RUN_CUSTOM_EXAMPLE=True to run this additional sweep."
)
custom_predictions